# **RNN과 자연어처리**


---



**(코랩에서)한글 폰트 지정하는 방법**

In [ ]:
# 코랩에서 실행
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import shutil
import warnings
import logging

# ── 1. 나눔 폰트 설치 ──────────────────────────────────────────
!apt-get install -y fonts-nanum > /dev/null 2>&1

# ── 2. matplotlib 캐시 디렉터리 삭제 ──────────────────────────
shutil.rmtree(mpl.get_cachedir(), ignore_errors=True)

# ── 3. 폰트 등록 및 유니코드 마이너스 경고 방지 ──────────────────
font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (6, 4)

# 수학 기호(마이너스 등)를 일반 텍스트로 처리하도록 설정하여 폰트 에러 차단
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.default'] = 'regular'

# mathtext 및 font_manager 관련 폰트 결함 경고 완전히 무시 설정
logging.getLogger('matplotlib.mathtext').setLevel(logging.ERROR)
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib.font_manager')



---



# **2. 단어 임베딩**: one-hot → Word2Vec

| Lab |  주제 | 발전사 위치 |
|---|---|---|
| 21 |  one-hot 벡터는 왜 "의미"를 모를까 | 1990s 이전 |
| 2-2 |  NSMC로 Word2Vec 직접 학습 · 유사어 · 유추 | 2013 Mikolov et al. |
| 2-3 | 임베딩 공간 2D 시각화 (5차시 PCA 복습) | |
| 2-4 |  영화 말뭉치 vs 게임 말뭉치: 같은 단어, 다른 이웃 | 분포 가설(1954) · 정적 임베딩의 한계 |

In [ ]:
%pip install -q kiwipiepy gensim koreanize-matplotlib

In [ ]:
import os, urllib.request, time, pickle
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib   # 한글 폰트
plt.rcParams['axes.unicode_minus'] = False   # 음수 기호 깨짐 방지
from kiwipiepy import Kiwi

URLS = {
    "ratings_train.txt": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
    "ratings_test.txt":  "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt",
    "steam.txt":         "https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/steam.txt",
}
for fn, url in URLS.items():
    if not os.path.exists(fn):
        urllib.request.urlretrieve(url, fn)
nsmc = pd.concat([pd.read_csv(f, sep="\t").dropna() for f in ["ratings_train.txt", "ratings_test.txt"]])
steam = pd.read_csv("steam.txt", sep="\t", header=None, names=["label", "document"], quoting=3).dropna()
os.makedirs("figs", exist_ok=True)

---
## **2-1. one-hot**: 모든 단어는 서로 "남남"이다

### **[실습] one-hot은 왜 '의미'를 모를까?**

In [ ]:
vocab = ["강아지", "멍멍이", "고양이", "자동차"]
onehot = np.eye(len(vocab))                     # 단어마다 자기 자리만 1
cos = lambda a, b: a @ b / (np.linalg.norm(a) * np.linalg.norm(b))   # 코사인 유사도
print("one-hot  강아지·멍멍이 유사도:", cos(onehot[0], onehot[1]))   # 0 → 의미가 같아도 0

# 밀집(dense) 벡터라면? (사람이 손으로 만든 예시 특성: [동물, 귀여움, 탈것])
dense = np.array([[0.9, 0.8, 0.0],
                [0.9, 0.9, 0.0],
                [0.9, 0.6, 0.0],
                [0.0, 0.1, 0.9]])
for i in range(1, 4):
    print(f"dense    강아지·{vocab[i]} 유사도: {cos(dense[0], dense[i]):.3f}")
# 👉 Word2Vec은 이 특성들을 사람 대신 '데이터(주변 단어)'로부터 학습한다

---
## **2-2. Word2Vec 학습** — "단어의 의미는 함께 쓰이는 단어가 결정한다"
Skip-gram 목표: 중심 단어 $w_t$로 주변 단어 $w_{t+j}$를 맞힌다 → $\max \sum_t \sum_{-c\le j\le c, j\ne0} \log P(w_{t+j}\mid w_t)$

### **[실습] NSMC 데이터로 Word2Vec 학습하기**
앞에서 사용한 NSMC 데이터(영화 리뷰 댓글)를 이용하여  Word2Vec 학습을 확인해 보세요.

In [ ]:
kiwi = Kiwi()
def kiwi_tok(texts, cache):
    if os.path.exists(cache):
        return pickle.load(open(cache, "rb"))
    toks = [[t.form for t in r] for r in kiwi.tokenize(texts)]
    pickle.dump(toks, open(cache, "wb"))
    return toks

t0 = time.time()
corpus_movie = kiwi_tok(nsmc.document.tolist(), "tok_nsmc_all.pkl")
corpus_game  = kiwi_tok(steam.document.tolist(), "tok_steam_all.pkl")
print(f"형태소 분석 완료 {time.time()-t0:.0f}초 | 영화 {len(corpus_movie):,}문장, 게임 {len(corpus_game):,}문장")
print(corpus_movie[0])

In [ ]:
from gensim.models import Word2Vec
t0 = time.time()
w2v = Word2Vec(corpus_movie, vector_size=100, window=5, min_count=5,
               sg=1,            # 1 = Skip-gram, 0 = CBOW
               negative=5,      # 네거티브 샘플링: 가짜 주변단어 5개와 구별하도록 학습
               epochs=5, workers=4, seed=42)
print(f"학습 {time.time()-t0:.0f}초 | 어휘 {len(w2v.wv):,}개 × {w2v.vector_size}차원")
w2v.wv.save("w2v_nsmc.kv")      # 재사용을 위해 저장

In [ ]:
for w in ["송강호", "재밌", "공포", "배우", "ㅋㅋ"]:
    print(f"{w:>4} →", [(x, round(s, 2)) for x, s in w2v.wv.most_similar(w, topn=6)])

### **[실습] 벡터 연산(유추)과 분포 가설의 함정**
앞에서 사용한 코드에 이어서 단어 벡터의 덧셈·뺄셈(유추)과 '최악'의 이웃 단어를 확인해 보세요.

In [ ]:
# 유추(analogy): 벡터 덧셈·뺄셈  a : b = c : ?
def analogy(a, b, c, k=3):
    return [(x, round(s, 2)) for x, s in w2v.wv.most_similar(positive=[b, c], negative=[a], topn=k)]
print("남자 : 여자 = 아빠 : ?", analogy("남자", "여자", "아빠"))
print("남자 : 여자 = 아들 : ?", analogy("남자", "여자", "아들"))
print("송강호 : 남자 = 전지현 : ?", analogy("송강호", "남자", "전지현"))

In [ ]:
# ⚠ 함께 쓰이는 단어 = 비슷한 단어? → '최악'의 이웃에 '최고'가 있다!
print("최악 →", [(x, round(s, 2)) for x, s in w2v.wv.most_similar("최악", topn=6)])
print("최악·최고 유사도:", round(float(w2v.wv.similarity("최악", "최고")), 3))
# "인생 최고의 영화" / "인생 최악의 영화" — 문맥이 같으면 반의어도 가까워진다 (분포 가설의 한계)

---
## **2-3. 임베딩 공간 들여다보기** (PCA 2D)

### **[실습] 임베딩 공간 들여다보기 (PCA 2D)**
앞에서 사용한 코드에 이어서 단어 벡터를 PCA로 2차원에 그려서 비슷한 단어끼리 모이는지 확인해 보세요.

In [ ]:
from sklearn.decomposition import PCA
groups = {
    "배우": ["송강호", "황정민", "하정우", "이병헌", "전지현", "김혜수"],
    "장르": ["공포", "스릴러", "코미디", "로맨스", "액션", "애니"],
    "긍정": ["최고", "명작", "감동", "재밌", "수작", "꿀잼"],
    "부정": ["최악", "쓰레기", "노잼", "지루", "졸작", "실망"],
}
words = [w for g in groups.values() for w in g if w in w2v.wv]
xy = PCA(n_components=2, random_state=0).fit_transform(np.stack([w2v.wv[w] for w in words]))
colors = dict(zip(groups, ["#004A80", "#0EA5B7", "#059669", "#DC2626"]))
plt.figure(figsize=(8, 5.2))
for g, ws in groups.items():
    idx = [words.index(w) for w in ws if w in words]
    plt.scatter(xy[idx, 0], xy[idx, 1], s=70, c=colors[g], label=g)
    for i in idx:
        plt.annotate(words[i], xy[i] + 0.04, fontsize=12)
plt.legend(fontsize=11); plt.title("NSMC Word2Vec 임베딩 (PCA 2차원)", fontsize=14); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("figs/w2v_pca.png", dpi=160); plt.show()

---
## **2-4. 말뭉치가 의미를 만든다** : 같은 단어, 다른 이웃  
분포 가설(Harris, 1954): 단어의 의미는 그 단어가 쓰인 **문맥의 분포**로 정해진다.

### **[실습] 말뭉치가 의미를 만든다**
앞에서 사용한 코드에 이어서 **말뭉치에 따라 같은 단어의 이웃이 어떻게 달라지는지** 확인해 보세요.

In [ ]:
w2v_game = Word2Vec(corpus_game, vector_size=100, window=5, min_count=5, sg=1, negative=5, epochs=5, workers=4, seed=42)
rows = []
for w in ["스토리", "캐릭터", "그래픽", "혜자", "타격감"]:
    nm = [x for x, _ in w2v.wv.most_similar(w, topn=4)] if w in w2v.wv else ["(어휘에 없음)"]
    ng = [x for x, _ in w2v_game.wv.most_similar(w, topn=4)] if w in w2v_game.wv else ["(어휘에 없음)"]
    rows.append([w, ", ".join(nm), ", ".join(ng)])
neighbor_table = pd.DataFrame(rows, columns=["단어", "영화 리뷰에서의 이웃", "게임 리뷰에서의 이웃"])
neighbor_table.to_csv("w2v_neighbors.csv", index=False)
neighbor_table

- [관찰]
    - '타격감'은 영화 말뭉치에서 드물어 이웃이 엉뚱하다 → **드문 단어는 벡터 품질이 낮다.**
    - 정적 임베딩은 '배(신체/선박)'를 구별하지 못한다 → **문맥 임베딩 필요**

In [ ]:
# 정적 임베딩의 한계: 문맥이 달라도 벡터는 하나뿐
for s in ["배가 아파서 영화 중간에 나왔다", "배가 침몰하는 장면이 압권"]:
    print(s, "→ '배' 벡터 앞 3개 값:", np.round(w2v.wv["배"][:3], 3))
print("👉 '배(신체)'와 '배(선박)'가 같은 벡터 → 문맥에 따라 달라지는 표현이 필요 (RNN → ELMo → BERT)")

### 정리
| 표현 | 차원 | 의미 유사성 | 문맥 반영 |
|---|---|---|---|
| one-hot / BoW | 어휘 크기(수만) | ✗ | ✗ |
| Word2Vec / FastText | 100~300 | ✓ | ✗ (단어당 벡터 1개) |
| RNN 은닉 상태 → ELMo → BERT | 수백~수천 | ✓ | ✓ |